In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Step 4 - Tissue Annotation (Unsupervised Macro-Structure Discovery)

Notebook counterpart of the SPACEMAP **Annotation** tab. Both drive the same backend
(`spacemap.tissue_annotation_utils`) and render the same diagnostics
(`spacemap.tissue_annotation_figures`), so a result produced here is identical to one
produced in the GUI with the same parameters.

**What this notebook does**

| Step | Purpose | Output |
|---|---|---|
| 1 | Virtual H&E optical-density tissue masking | `*_tissue_boundary.geojson` |
| 2 | Out-of-focus, flat-bright and filamentous lint removal | `*_valid_tissue.geojson`, `*_detected_lint.geojson` |
| 3 | Marker normalization, pixel clustering, merge clusters into domains | `*_raw_macro_clusters.geojson`, `*_valid_tissue_from_clusters_highres.geojson`, `*_macro_domains.geojson` |
| 4 | Pool domains into higher-level polygons and add buffer zones | `*_zones.geojson` |

**QuPath round-trips.** Every step reads its input GeoJSON from disk. Open any intermediate
file in QuPath, refine it, save it under the same name, and re-run the next step - your
edits are used, never a cached mask.

**Performance.** All image reads go through `PyramidImageReader`, which reads the requested
OME-TIFF pyramid level directly instead of decoding full resolution and striding. That is
both faster and anti-aliased.

Cell-level annotation (mapping these tissue annotations onto segmented cells) lives in
`Step4B_cell_annotation.ipynb`.

## 1. Setup

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Import this checkout's sources first: the environment may hold an editable
# install of a different SPACEMAP checkout.
project_root = Path("..").resolve()
if not (project_root / "src" / "spacemap").is_dir():
    project_root = Path(".").resolve()
src_path = str(project_root / "src")
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from spacemap import tissue_annotation_figures as taf
from spacemap import tissue_annotation_utils as ta

print("Project root:", project_root)
print("Backend     :", Path(ta.__file__).resolve())

## 2. Configuration

`ROI_DIR` is the ROI Extraction output directory and `OUTPUT_DIR` matches the tab's
"Tissue annotation output directory". The four configuration dictionaries are the same ones
the GUI edits and stores in `<project>.json`, so you can copy them between notebook and tab.

In [ ]:
# ----- Paths -----
ROI_DIR = f"{DATA_DIR}/example_study/registration/roi"
OUTPUT_DIR = os.path.join(ROI_DIR, "tissue_annotation")
POOL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "zones")
SAMPLE_ID = "P28_roi_1"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(POOL_OUTPUT_DIR, exist_ok=True)

FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures", SAMPLE_ID)
os.makedirs(FIGURE_DIR, exist_ok=True)
SAVE_FIGURES = True

MICRONS_PER_PIXEL = 0.5083  # full-resolution pixel size

# ----- Step 1: tissue masking -----
# Channels that define tissue. Leave both empty to detect the virtual H&E layers
# by name and use every remaining channel as a marker.
HE_CHANNELS = []              # e.g. ["HE_Red", "HE_Green", "HE_Blue"]
TISSUE_MARKER_CHANNELS = []   # e.g. ["DAPI"] for marker-based detection

tissue_config = dict(
    ta.DEFAULT_TISSUE_CONFIG,
    blur_downsample=2,             # pyramid level used for masking and artifacts
    microns_per_pixel=MICRONS_PER_PIXEL,
    # he_only | he_with_codex_edge_rescue | he_codex_union | codex_only
    # Use codex_only when the image has no virtual H&E layers: tissue is then
    # thresholded from the marker projection (DAPI alone works well).
    tissue_mask_mode="he_only",
    codex_threshold_mode="otsu",   # otsu adapts per image; fixed uses tissue_threshold
    he_od_threshold_factor=0.85,   # <1 is more inclusive, >1 is stricter
    tissue_morph_radius=11,
    min_tissue_island_px=5000,
    tissue_border_exclusion_um=20.0,
)

# ----- Step 2: artifacts -----
artifact_config = dict(
    ta.DEFAULT_ARTIFACT_CONFIG,
    blur_ratio_threshold=0.35,     # CODEX focus / H&E focus below this is out of focus
    detect_bright_artifacts=True,
    detect_filamentous_lint=True,
    lint_robust_z_threshold=12.0,  # brightness (robust Z) needed to seed a fibre
    lint_min_width_microns=5.0,    # the most effective control against thin scan seams
)

# ----- Step 3: pixel classification -----
cluster_config = dict(
    ta.DEFAULT_CLUSTERING_CONFIG,
    clustering_downsample=4,       # ~2.03 um per raster pixel at 0.5083 um/px
    microns_per_pixel=MICRONS_PER_PIXEL,
    arcsinh_mode="dynamic",        # per-marker background noise estimation
    use_top_k_masking=True,
    top_k_mode="auto",             # elbow of the mean sorted Z-score profile
    n_clusters=20,
    kmeans_fit_sample=0,           # >0 fits centroids on a subsample, then assigns all pixels
    fine_export_smooth_sigma=2.0,
)

# ----- Step 4: pooling and zones -----
pool_config = dict(ta.DEFAULT_POOLING_CONFIG)

def show(figure, name):
    '''Display a figure and, when enabled, save it next to the GeoJSON outputs.'''
    if SAVE_FIGURES and figure is not None:
        figure.savefig(os.path.join(FIGURE_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")
        figure.savefig(os.path.join(FIGURE_DIR, f"{name}.pdf"), bbox_inches="tight")
    if figure is not None:
        plt.show()

image_path = os.path.join(ROI_DIR, f"{SAMPLE_ID}.ome.tiff")
assert os.path.exists(image_path), f"Image not found: {image_path}"
print("Sample :", SAMPLE_ID)
print("Image  :", image_path)
print("Output :", OUTPUT_DIR)

## 3. Inspect the image pyramid

`PyramidImageReader` detects the on-disk layout once. `channel_series` means each channel is
stored as its own series followed by its pyramid levels (what the SPACEMAP registration and
ROI writers produce); `single_series` and `pages` are handled as fallbacks.

In [ ]:
reader = ta.PyramidImageReader(image_path)
detected_he, detected_codex = ta.split_he_and_codex_channels(reader.channel_names)

# Manual overrides win; otherwise use what was detected by name.
he_indices = (
    [reader.channel_names.index(name) for name in HE_CHANNELS]
    if HE_CHANNELS else list(detected_he)
)
codex_indices = (
    [reader.channel_names.index(name) for name in TISSUE_MARKER_CHANNELS]
    if TISSUE_MARKER_CHANNELS else list(detected_codex)
)

print(f"Layout          : {reader.layout}")
print(f"Channels        : {reader.n_channels}")
print(f"Pyramid levels  : {reader.level_count}")
for level, (height, width) in enumerate(reader.level_shapes):
    print(f"  Level {level}: {height} x {width} ({height * width:,} px)")
print(f"H&E channels    : {[reader.channel_names[i] for i in he_indices] or 'none detected'}")
print(f"Marker channels : {len(codex_indices)}")
if len(he_indices) < 3 and tissue_config["tissue_mask_mode"] != "codex_only":
    print("  NOTE: fewer than three H&E layers. Either set HE_CHANNELS above, or use")
    print("        tissue_mask_mode='codex_only' with TISSUE_MARKER_CHANNELS=['DAPI'].")

for factor in (2, 4, 8):
    print(f"Read at {factor}x -> {reader.target_shape(factor)}")

## 4. Step 1 - Detect valid tissue from H&E

Each virtual H&E channel is normalized against its own white-background reference, converted
to optical density and combined into a stain-strength map before Otsu thresholding. Zero-valued
acquisition borders are excluded explicitly, so black scanner frames and CODEX-only lint
cannot enter the tissue mask.

**Only the three H&E channels are read here.** In `he_only` mode the CODEX markers do not
affect the mask, so decoding them would be wasted work; on a 40-channel ROI that is the
difference between about 6 seconds and about 30. The marker channels are read once, in
step 2, where the artifact detectors actually need them. Set
`tissue_mask_mode="he_with_codex_edge_rescue"` or `"he_codex_union"` and the marker pass
happens here automatically.

In [ ]:
tissue = ta.detect_tissue_mask(
    reader, tissue_config, he_indices=he_indices, codex_indices=codex_indices
)

stats = tissue["stats"]
print(f"Mode            : {stats['mode']}")
print(f"Tissue coverage : {stats['tissue_percent']:.1f}% ({stats['tissue_pixels']:,} px at {tissue['downsample']}x)")
if stats["he_od_threshold"] is not None:
    print(f"OD threshold    : {stats['he_od_threshold']:.4f} (Otsu {stats['otsu']:.4f})")
    print(f"White references: {stats['he_white_references']}")
if stats.get("codex_threshold") is not None:
    print(f"Marker threshold: {stats['codex_threshold']:.2f} ({stats['codex_threshold_mode']})")

show(taf.tissue_mask_figure(tissue, SAMPLE_ID), "step1_tissue_mask")

In [ ]:
boundary_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_tissue_boundary.geojson")
boundary_features = ta.mask_to_geojson_features(
    tissue["tissue_mask"],
    scale_factor=float(tissue_config["blur_downsample"]),
    class_name="Tissue_Boundary",
    color_rgb=-16711936,
    simplify_px=float(tissue_config["boundary_simplify_px"]),
    merge=False,
    locked=True,
)
size_mb = ta.write_feature_collection(boundary_features, boundary_path)
print(f"Tissue boundary GeoJSON: {boundary_path} ({len(boundary_features)} features, {size_mb:.2f} MB)")

> **Review point.** Open `*_tissue_boundary.geojson` in QuPath, correct it if needed, and save it
> under the same name. Step 2 rasterizes whatever is on disk.

## 5. Step 2 - Remove blur, bright artifacts and lint

Three independent artifact classes are detected while true tissue is preserved:

1. **Out-of-focus tissue** from normalized Laplacian focus loss relative to the H&E reference.
2. **Flat bright artifacts** - bright *and* texture-free - detected conservatively inside tissue.
3. **Filamentous lint**, seeded only in H&E-negative space from the strongest robust signal in
   any single CODEX channel, then filtered by physical length, width, aspect ratio,
   filamentarity and seam geometry.

Only the lint that overlaps tissue is subtracted; all lint is exported separately for review.

*Precision-first limitation:* a fibre entirely enclosed in H&E-positive tissue, with no external
H&E-negative segment, is not removed automatically, because its shape alone cannot safely
distinguish it from vessels, glands or stromal tracks. Review those cases in QuPath using
`*_detected_lint.geojson`.

In [ ]:
# Markers that seed artifact detection. Virtual H&E layers are excluded by default;
# add them here if you want H&E intensity itself to seed artifacts.
artifact_marker_indices = list(codex_indices)

# Step 1 read only the three H&E channels, because in he_only mode the markers
# contribute nothing to the mask. The artifact detectors *do* need the per-marker
# projection and robust-Z maps, so this pass requests them explicitly - this is
# the one place the marker channels are decoded.
maps = ta.detect_tissue_mask(
    reader, tissue_config, he_indices=he_indices, codex_indices=artifact_marker_indices,
    need_codex_maps=True,
)

# Reload the reviewed boundary from disk so QuPath edits are honoured.
tissue_mask = ta.load_geojson_mask(
    boundary_path, maps["shape"], float(tissue_config["blur_downsample"])
)
print(f"Reloaded tissue mask: {tissue_mask.sum():,} px ({100.0 * tissue_mask.mean():.1f}% of image)")

artifacts = ta.detect_artifacts(
    tissue_mask=tissue_mask,
    codex_projection=maps["codex_projection"],
    he_gray=maps["he_gray"],
    codex_robust_max=maps["codex_robust_max"],
    channel_positive_count=maps["channel_positive_count"],
    n_codex_channels=len(artifact_marker_indices),
    config=artifact_config,
    microns_per_pixel=MICRONS_PER_PIXEL,
    downsample=int(tissue_config["blur_downsample"]),
    border_px=int(maps["border_px"]),
    # Per-marker evidence: lets the detector see defocus that hit only some
    # cycles, and names the markers behind each region in the export below.
    marker_names=maps["codex_channel_names"],
    channel_focus_grid=maps["channel_focus_grid"],
    channel_signal_grid=maps["channel_signal_grid"],
    sample_id=SAMPLE_ID,
)

stats = artifacts["stats"]
print(f"Blurry            : {stats['blurry_percent']:.2f}% of tissue")
print(f"Bright / flat     : {stats['bright_percent']:.2f}% of tissue")
print(f"Lint overlap      : {stats['lint_tissue_percent']:.2f}% of tissue ({stats['accepted_lint']:,} accepted fibres)")
print(f"Clean valid tissue: {stats['valid_percent']:.2f}% ({stats['valid_pixels']:,} px)")

if not artifacts["lint_components"].empty:
    display(artifacts["lint_components"].head(20))

show(
    taf.artifact_figure(
        tissue_mask, artifacts, maps["codex_robust_max"], maps["he_gray"],
        maps["codex_projection"], float(artifact_config["lint_robust_z_threshold"]), SAMPLE_ID,
    ),
    "step2_artifacts",
)

In [ ]:
scale = float(tissue_config["blur_downsample"])
simplify = float(artifact_config["boundary_simplify_px"])

# One file per artifact class, plus their union and the cleaned tissue, so each
# can be reviewed or corrected independently in QuPath. Colours match the figure.
ARTIFACT_EXPORTS = [
    ("valid_tissue_mask", "valid_tissue",       "Valid_Tissue",       "#2ecc71", simplify),
    ("blur_mask",         "detected_blur",      "Detected_Blur",      "#ffff00", simplify),
    ("bright_mask",       "detected_bright",    "Detected_Bright",    "#ff00ff", simplify),
    ("lint_mask",         "detected_lint",      "Detected_Lint",      "#00ffff", max(simplify / 2, 0.5)),
    ("artifact_mask",     "detected_artifacts", "Detected_Artifacts", "#ff5252", simplify),
]

artifact_paths = {}
for mask_key, suffix, class_name, hex_color, tolerance in ARTIFACT_EXPORTS:
    mask = artifacts[mask_key]
    path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_{suffix}.geojson")
    features = (
        ta.mask_to_geojson_features(
            mask, scale, class_name,
            color_rgb=int(ta.qupath_color_from_hex(hex_color)),
            simplify_px=tolerance,
        )
        if mask is not None and mask.any() else []
    )
    # Always write, even when empty, so a stale file cannot be mistaken for a result.
    size_mb = ta.write_feature_collection(features, path)
    artifact_paths[suffix] = path
    coverage = 100.0 * float(mask.mean()) if mask is not None else 0.0
    print(f"{class_name:20s} {coverage:6.2f}% of image  {size_mb:6.2f} MB  {os.path.basename(path)}")

valid_tissue_path = artifact_paths["valid_tissue"]
lint_path = artifact_paths["detected_lint"]

> **Review point.** Refine `*_valid_tissue.geojson` in QuPath if needed. Step 3 rasterizes the file
> from disk at the clustering resolution.

## 6. Step 3a - Feature stack over the reviewed valid tissue

Per-marker dynamic arcsinh (cofactor `max(floor, k x sigma_background)`), robust percentile
scaling, light spatial smoothing, Z-scoring across tissue pixels, and optional Top-K marker
masking. Setting `top_k_mode="auto"` picks K from the elbow of the mean sorted Z-score profile,
adapting it to how many markers are typically co-enriched in this sample and panel.

The transformations run in place on one `(C, H, W)` float32 array, so peak memory is about a
third of a naive implementation.

In [ ]:
CLUSTER_MARKER_INDICES = list(codex_indices)   # drop uninformative markers here for cleaner domains

factor = int(cluster_config["clustering_downsample"])
grid_shape = reader.target_shape(factor)
valid_mask = ta.load_geojson_mask(valid_tissue_path, grid_shape, float(factor))
print(f"Clustering grid: {grid_shape[0]} x {grid_shape[1]} px, valid tissue {valid_mask.sum():,} px")

feature_stack = ta.build_feature_stack(reader, CLUSTER_MARKER_INDICES, valid_mask, cluster_config)
print(f"Feature matrix : {feature_stack['features'].shape[0]:,} px x {feature_stack['features'].shape[1]} markers")
print(f"Top-K used     : {feature_stack['top_k_effective']} (automatic estimate {feature_stack['top_k_auto']})")

reference_channel = reader.read_channels(CLUSTER_MARKER_INDICES[:1], factor)[0]
show(taf.valid_tissue_sync_figure(reference_channel, valid_mask, factor, SAMPLE_ID), "step3_valid_tissue_sync")
show(taf.intensity_distribution_figure(feature_stack, SAMPLE_ID), "step3_intensity_distributions")
show(taf.top_k_elbow_figure(feature_stack, cluster_config, SAMPLE_ID), "step3_topk_elbow")

pd.DataFrame(sorted(feature_stack["cofactors"].items()), columns=["Marker", "Arcsinh cofactor"])

## 7. Step 3b - Cluster and characterise

Three complementary views of the same clusters:

* **Absolute intensity** - what each cluster is bright in.
* **Mean Z-score** - what each cluster is *enriched* in relative to the rest of the tissue.
* **Size-aware balanced** - additionally shrinks unreliable small clusters toward the tissue-wide
  marker mean and equalises colour contrast per marker, so neither a 0.2% cluster nor a dominant
  one distorts the palette. This is the view the GUI uses for click-to-merge, and its row
  dendrogram gives the cluster ordering used below.

In [ ]:
clustering = ta.run_pixel_clustering(feature_stack, cluster_config)
profiles = ta.build_cluster_profiles(feature_stack, clustering, cluster_config)

print(f"Clusters fitted: {clustering['n_clusters']}")
display(profiles["size_summary"].sort_values("tissue_percent", ascending=False))

# Spatial layout of the raw clusters, available before any merging decision.
show(
    taf.cluster_map_figure(
        clustering["cluster_map"], clustering["cluster_colors"],
        profiles["size_summary"], SAMPLE_ID, clustering["reserved_low_signal_cluster"],
    ),
    "step3_cluster_map",
)

signature_figures = taf.cluster_signature_figures(profiles)
show(signature_figures["absolute"], "step3_cluster_absolute")
show(signature_figures["z_score"], "step3_cluster_zscore")
show(signature_figures["balanced"], "step3_cluster_balanced")

print("Cluster order from the balanced dendrogram (similar signatures adjacent):")
print(signature_figures["cluster_order"])

print("\nTop-3 size-aware enriched markers per cluster:")
for cluster_id in profiles["shrunk_profiles"].index:
    top = profiles["shrunk_profiles"].loc[cluster_id].nlargest(3)
    share = profiles["size_summary"].loc[cluster_id, "tissue_percent"]
    reliability = profiles["size_summary"].loc[cluster_id, "reliability"]
    markers = ", ".join(f"{marker} ({value:+.2f})" for marker, value in top.items())
    print(f"  Cluster {cluster_id:2d} ({share:5.2f}% tissue, r={reliability:.2f}): {markers}")

## 8. Step 3c - Merge clusters into biological domains

Fill in `CLUSTER_TO_DOMAIN` using the heatmaps above. In the GUI this same mapping is built by
clicking columns of the balanced heatmap and applying a domain to the whole selection.

`EXCLUDED_CLUSTERS` lists cluster IDs that turned out to be residual lint or debris: they are
dropped from the rebuilt valid-tissue polygon and from the domain export.

In [ ]:
CLUSTER_TO_DOMAIN = {cluster_id: "Stroma" for cluster_id in profiles["balanced_scores"].index}
# Example of a curated mapping - replace with your own reading of the heatmaps:
# CLUSTER_TO_DOMAIN.update({0: "Epithelia", 3: "Epithelia", 8: "Epithelia", 19: "Endothelia"})

EXCLUDED_CLUSTERS = []          # e.g. [3, 7]
DOMAIN_COLORS = {}              # e.g. {"Epithelia": "#e41a1c", "Stroma": "#377eb8"}

cluster_export = ta.export_cluster_partition(
    clustering, feature_stack, cluster_config, OUTPUT_DIR, SAMPLE_ID,
    excluded_clusters=EXCLUDED_CLUSTERS,
)
for message in cluster_export["retries"]:
    print(" ", message)
print(f"Cluster GeoJSON      : {cluster_export['clusters_path']} ({cluster_export['clusters_size_mb']:.2f} MB)")
print(f"Rebuilt valid tissue : {cluster_export['merged_valid_path']} ({cluster_export['merged_valid_size_mb']:.2f} MB)")
print(f"Raw polygons         : {cluster_export['polygon_count']:,} at sigma={cluster_export['smoothing_sigma']:.2f}")
print(f"Export resolution    : {cluster_export['export_resolution_um']:.3f} um/pixel, shared boundaries")

show(
    taf.cluster_partition_figure(
        clustering["cluster_map"], cluster_export["export_label_map"],
        clustering["cluster_colors"],
        f"fine gapless partition (sigma={cluster_export['smoothing_sigma']:.2f})", SAMPLE_ID,
    ),
    "step3_cluster_partition",
)

In [ ]:
domain_map = {
    cluster_id: domain
    for cluster_id, domain in CLUSTER_TO_DOMAIN.items()
    if domain and cluster_id not in EXCLUDED_CLUSTERS
}

domain_export = ta.export_domain_partition(
    cluster_export, domain_map, feature_stack, clustering, cluster_config,
    OUTPUT_DIR, SAMPLE_ID, domain_colors=DOMAIN_COLORS,
)
print(f"Domain GeoJSON: {domain_export['domains_path']} ({domain_export['size_mb']:.2f} MB)")
print(f"Domains       : {domain_export['domain_order']}")

# Preview the merge straight from the cluster map, without vectorizing.
show(
    taf.domain_preview_figure(
        clustering["cluster_map"], domain_map,
        domain_export["domain_colors"], SAMPLE_ID,
    ),
    "step3_merge_preview",
)

domain_figures = taf.domain_signature_figures(domain_export)
show(domain_figures["absolute"], "step3_domain_absolute")
show(domain_figures["z_score"], "step3_domain_zscore")
show(taf.domain_map_figure(domain_export, SAMPLE_ID), "step3_domain_map")

> **Review point.** Refine `*_macro_domains.geojson` in QuPath - split, merge or re-class domains -
> and save under the same name before running step 4.

## 9. Step 4 - Pool domains and create zones

Pooling unions several domains into one new named polygon while keeping the originals, which is
how a single `tissue` annotation is produced from Epithelia + Stroma + Endothelia. Zones then add
inner/outer buffer rings around chosen compartments; zones at the same level are merged across all
source labels, so `outer_1` is one consistent annotation for the cell-annotation step.

In [ ]:
domains_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_ID}_macro_domains.geojson")
print("Classes on disk:", ta.read_geojson_class_names(domains_path))

POOL_RULES = [
    {"name": "tissue", "sources": ["Epithelia", "Stroma", "Endothelia"], "subtract": []},
]
ZONE_RULES = [
    {"label": "Epithelia", "width_px": 100, "inner": 1, "outer": 2},
]
ZONE_CLIP_CLASS = "tissue"     # outer zones never extend past this annotation
KEEP_SOURCE_CLASSES = True

zoned_path = os.path.join(POOL_OUTPUT_DIR, f"{SAMPLE_ID}_zones.geojson")

pool_result = ta.pool_geojson_classes(
    domains_path, POOL_RULES, zoned_path,
    keep_source_classes=KEEP_SOURCE_CLASSES, config=pool_config, class_colors=DOMAIN_COLORS,
)
display(pd.DataFrame(pool_result["created"]))

zone_features = ta.generate_zone_features(
    zoned_path, ZONE_RULES, clip_class=ZONE_CLIP_CLASS, config=pool_config
)
if zone_features:
    features = ta._iter_geojson_features(zoned_path)
    features.extend(zone_features)
    ta.write_feature_collection(features, zoned_path)

print(f"\nFinal annotation: {zoned_path}")
print("Classes written :", ta.read_geojson_class_names(zoned_path))

## 10. Batch processing

Steps 1, 2 and 4 are deterministic given their parameters and run well in batch. Step 3 stays
per-image because cluster identities and merge decisions are image-specific - the same is true in
the GUI, where only steps 1, 2 and 4 expose a batch Run button.

In [ ]:
BATCH_SAMPLES = []   # e.g. ["P11_roi_1", "P11_roi_2"]; leave empty to skip

for sample in BATCH_SAMPLES:
    path = os.path.join(ROI_DIR, f"{sample}.ome.tiff")
    if not os.path.exists(path):
        print(f"{sample}: image not found, skipped")
        continue
    print(f"\n=== {sample} ===")
    batch_reader = ta.PyramidImageReader(path)
    batch_he, batch_codex = ta.split_he_and_codex_channels(batch_reader.channel_names)

    # need_codex_maps=True because the artifact step below reuses these maps.
    batch_tissue = ta.detect_tissue_mask(batch_reader, tissue_config, need_codex_maps=True)
    batch_boundary = os.path.join(OUTPUT_DIR, f"{sample}_tissue_boundary.geojson")
    ta.write_feature_collection(
        ta.mask_to_geojson_features(
            batch_tissue["tissue_mask"], float(tissue_config["blur_downsample"]),
            "Tissue_Boundary", color_rgb=-16711936, merge=False, locked=True,
        ),
        batch_boundary,
    )
    print(f"  tissue {batch_tissue['stats']['tissue_percent']:.1f}%")

    batch_mask = ta.load_geojson_mask(
        batch_boundary, batch_tissue["shape"], float(tissue_config["blur_downsample"])
    )
    batch_artifacts = ta.detect_artifacts(
        tissue_mask=batch_mask,
        codex_projection=batch_tissue["codex_projection"],
        he_gray=batch_tissue["he_gray"],
        codex_robust_max=batch_tissue["codex_robust_max"],
        channel_positive_count=batch_tissue["channel_positive_count"],
        n_codex_channels=len(batch_codex),
        config=artifact_config,
        microns_per_pixel=MICRONS_PER_PIXEL,
        downsample=int(tissue_config["blur_downsample"]),
        border_px=int(batch_tissue["border_px"]),
        marker_names=batch_tissue["codex_channel_names"],
        channel_focus_grid=batch_tissue["channel_focus_grid"],
        channel_signal_grid=batch_tissue["channel_signal_grid"],
        sample_id=sample,
    )
    for mask_key, suffix, class_name, hex_color, tolerance in ARTIFACT_EXPORTS:
        mask = batch_artifacts[mask_key]
        ta.write_feature_collection(
            ta.mask_to_geojson_features(
                mask, float(tissue_config["blur_downsample"]), class_name,
                color_rgb=int(ta.qupath_color_from_hex(hex_color)), simplify_px=tolerance,
            ) if mask is not None and mask.any() else [],
            os.path.join(OUTPUT_DIR, f"{sample}_{suffix}.geojson"),
        )
    print(f"  valid tissue {batch_artifacts['stats']['valid_percent']:.2f}%")

## 11. Standalone annotation inspector

Self-contained: point it at any GeoJSON and its image to recompute domain signatures and the
spatial map. Useful for reviewing a hand-drawn or QuPath-refined annotation without re-running
the pipeline.

In [ ]:
INSPECT_GEOJSON = os.path.join(POOL_OUTPUT_DIR, f"{SAMPLE_ID}_zones.geojson")
INSPECT_IMAGE = image_path
INSPECT_DOWNSAMPLE = 8

inspect_reader = ta.PyramidImageReader(INSPECT_IMAGE)
_, inspect_markers = ta.split_he_and_codex_channels(inspect_reader.channel_names)
inspect_shape = inspect_reader.target_shape(INSPECT_DOWNSAMPLE)

class_names = ta.read_geojson_class_names(INSPECT_GEOJSON)
print(f"{len(class_names)} classes: {class_names}")

# Rasterize one class at a time, then build a single label map.
label_map = np.full(inspect_shape, -1, dtype=np.int16)
for class_index, class_name in enumerate(class_names):
    class_mask = ta.load_geojson_mask(
        INSPECT_GEOJSON, inspect_shape, float(INSPECT_DOWNSAMPLE), class_names=[class_name]
    )
    label_map[class_mask] = class_index

covered = int((label_map >= 0).sum())
print(f"Coverage: {covered:,} px ({100.0 * covered / label_map.size:.1f}% of the grid)")

stack = inspect_reader.read_channels(inspect_markers, INSPECT_DOWNSAMPLE)
marker_names = [inspect_reader.channel_names[i] for i in inspect_markers]
pixels = np.arcsinh(stack.reshape(len(inspect_markers), -1).T / 5.0)
z_scores = (pixels - pixels.mean(axis=0)) / np.where(pixels.std(axis=0) > 0, pixels.std(axis=0), 1.0)

flat_labels = label_map.reshape(-1)
selected = flat_labels >= 0
labels = [class_names[index] for index in flat_labels[selected]]

absolute_profiles = pd.DataFrame(pixels[selected], columns=marker_names).assign(domain=labels).groupby("domain").mean()
z_profiles = pd.DataFrame(z_scores[selected], columns=marker_names).assign(domain=labels).groupby("domain").mean()

inspect_result = {
    "domain_profiles": absolute_profiles,
    "domain_z_profiles": z_profiles,
    "domain_label_map": label_map,
    "domain_order": class_names,
    "domain_colors": {name: ta.deterministic_class_hex_color(name) for name in class_names},
}
inspect_figures = taf.domain_signature_figures(inspect_result)
show(inspect_figures["absolute"], "inspector_domain_absolute")
show(inspect_figures["z_score"], "inspector_domain_zscore")
show(taf.domain_map_figure(inspect_result, os.path.basename(INSPECT_GEOJSON)), "inspector_domain_map")

## Summary

| File | Produced by | Consumed by |
|---|---|---|
| `*_tissue_boundary.geojson` | Step 1 | Step 2 (re-read from disk) |
| `*_valid_tissue.geojson` | Step 2 | Step 3 (re-read from disk) |
| `*_detected_blur.geojson` | Step 2 | QuPath review |
| `*_detected_bright.geojson` | Step 2 | QuPath review |
| `*_detected_lint.geojson` | Step 2 | QuPath review |
| `*_detected_artifacts.geojson` | Step 2 | QuPath review (union of the three above) |
| `*_raw_macro_clusters.geojson` | Step 3 | QuPath review |
| `*_valid_tissue_from_clusters_highres.geojson` | Step 3 | Optional re-entry into step 3 |
| `*_macro_domains.geojson` | Step 3 | Step 4 (re-read from disk) |
| `*_zones.geojson` | Step 4 | `Step4B_cell_annotation.ipynb` and the Annotation tab's cell annotation subtab |

Figures and tables are written to `<OUTPUT_DIR>/figures/<SAMPLE_ID>/` as PNG plus PDF, matching
what the GUI saves for the same run.